# Analisis aktivitas petir tahunan pada 617 tower

Data: 617 tower, 2020-2025. Target: jumlah petir di area tower (`Count`), bukan sambaran langsung atau gangguan transmisi.

## 1. Tujuan
Perkirakan `Count` 2025 dari data yang tersedia sampai 2024, lalu cek hasilnya.

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from openpyxl import load_workbook

from ta_ml.cleaning import (
    RAW_GEOMETRY_FILENAME, RAW_LIGHTNING_FILENAME, RAW_STATIC_FILENAME,
    build_clean_workbook,
)
from ta_ml.config import COUNT_DENSITY_DIVISOR, TOWER_COUNT
from ta_ml.evaluation import evaluate_models
from ta_ml.panel import build_model_panel

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA_DIR = Path(os.environ.get('TA_ML_DATA_DIR', ROOT / 'data')).resolve()
OUTPUT_DIR = Path(os.environ.get('TA_ML_OUTPUT_DIR', ROOT / 'data' / 'output')).resolve()
required_files = [RAW_LIGHTNING_FILENAME, RAW_STATIC_FILENAME, RAW_GEOMETRY_FILENAME]
missing = [name for name in required_files if not (DATA_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Workbook asli belum tersedia di {DATA_DIR}: {missing}')
print('Data lokal:', DATA_DIR)
print('Hasil lokal:', OUTPUT_DIR)

## 2. Sumber data

In [ ]:
sources = [
    (RAW_LIGHTNING_FILENAME, 'PKY-DATA PETIR', 'riwayat Count 2020-2025'),
    (RAW_STATIC_FILENAME, 'Kasongan-SKS', 'contoh sheet snapshot atribut 2026'),
    (RAW_GEOMETRY_FILENAME, 'INPUT_TOWER', 'geometri dan input PLN'),
]
source_rows = []
for filename, sheet_name, role in sources:
    workbook = load_workbook(DATA_DIR / filename, read_only=True, data_only=True)
    try:
        source_rows.append({'file': filename, 'sheet': sheet_name, 'baris_sheet': workbook[sheet_name].max_row, 'peran': role})
    finally:
        workbook.close()
display(pd.DataFrame(source_rows))

## 3. Cleaning dan validasi

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
clean = build_clean_workbook(
    source_dir=DATA_DIR, output_path=OUTPUT_DIR / 'ta_ml_clean_v2.xlsx', overwrite=True
)
history = pd.read_excel(clean.output_path, sheet_name='tower_year')
registry = pd.read_excel(clean.output_path, sheet_name='tower_registry')
audit_2020 = pd.read_excel(clean.output_path, sheet_name='coordinate_2020_audit')
source_notes = pd.read_excel(clean.output_path, sheet_name='source_notes')
print(f'{len(history):,} baris tower-tahun; {len(registry)} tower dalam registry')
display(source_notes[['source', 'time_scope', 'join_or_usage_rule']])

### 3.1. Cocokkan identitas tower 2020 lewat GPS

In [ ]:
display(audit_2020['match_status'].value_counts().rename_axis('status').reset_index(name='baris'))
display(audit_2020[['source_tower_key', 'matched_tower_key', 'source_count_2020', 'match_status']].head(12))
coverage = history.groupby('year').agg(baris=('tower_key', 'size'), count_terisi=('count', 'count'), tower_unik=('tower_key', 'nunique')).reset_index()
display(coverage)
assert len(history) == TOWER_COUNT * 6 and len(registry) == TOWER_COUNT
assert not history.duplicated(['corridor', 'tower_key', 'year']).any()
assert (coverage['baris'] == TOWER_COUNT).all()

### 3.2. Cek nilai, geometri, dan sumbernya

In [ ]:
valid_density = history.dropna(subset=['count', 'density'])
density_matches = (valid_density['count'] / COUNT_DENSITY_DIVISOR).round(2).eq(valid_density['density'])
print(f'Count/3,1273 cocok dengan Density: {density_matches.sum()}/{len(valid_density)} baris')
print('Nilai kolom Area sumber:', history['area_km2'].dropna().unique().tolist())
display(registry[['corridor', 'tower_name', 'geometry_source_sheet', 'geometry_source_row', 'elevation_m', 'tower_height_m']].head(8))
print('Geometri kosong:', registry[['elevation_m', 'tower_height_m', 'span_left_m', 'span_right_m']].isna().sum().to_dict())
assert density_matches.all()

## 4. Gambaran data tiap tahun dan koridor

In [ ]:
display(history.groupby('year')['count'].agg(['count', 'mean', 'median', 'min', 'max']).round(2))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
history.boxplot(column='count', by='year', ax=axes[0], grid=False)
axes[0].set(title='Count petir per tahun', xlabel='Tahun', ylabel='Count')
history.boxplot(column='count', by='corridor', ax=axes[1], grid=False)
axes[1].set(title='Count petir per koridor', xlabel='Koridor', ylabel='Count')
fig.suptitle('')
plt.tight_layout()
plt.show()

## 5. Fitur dan pembagian tahun
2020 untuk riwayat awal; 2021-2024 untuk memilih dan melatih; 2025 untuk mengecek hasil.

In [ ]:
panel_result = build_model_panel(clean_path=clean.output_path, output_path=OUTPUT_DIR / 'ta_ml_model_panel.csv', overwrite=True)
panel = pd.read_csv(panel_result.output_path)
display(panel[['corridor', 'tower_key', 'feature_year', 'target_year', 'split', 'count_lag1', 'count_lag2', 'target_count']].head(8))
display(panel.groupby(['target_year', 'split']).size().rename('baris').reset_index())
assert len(panel) == TOWER_COUNT * 5 and (panel['target_year'] == panel['feature_year'] + 1).all()

## 6. Pilih model dengan data 2023-2024

In [ ]:
evaluation = evaluate_models(panel_path=panel_result.output_path, pln_path=DATA_DIR / RAW_GEOMETRY_FILENAME, output_dir=OUTPUT_DIR, overwrite=True)
report = json.loads(evaluation.report_path.read_text(encoding='utf-8'))
predictions = pd.read_csv(evaluation.predictions_path)
candidates = pd.DataFrame(report['validation_candidates']['without_grounding'])
display(candidates[['model', 'feature_group', 'mean_mae']].sort_values('mean_mae').round(2))
selected = next(row for row in report['validation_candidates']['without_grounding'] if row['model'] == report['selected_model'] and row['feature_group'] == report['selected_feature_group'])
fold_rows = []
for fold in report['selection_folds']:
    train_fold = panel[panel['target_year'].isin(fold['train_target_years'])]
    valid_fold = panel[panel['target_year'] == fold['validation_target_year']]
    fold_rows.append({
        'tahun_dinilai': fold['validation_target_year'],
        'mae_model': next(score['mae'] for score in selected['fold_scores'] if score['validation_target_year'] == fold['validation_target_year']),
        'mae_tebak_median_latih': (valid_fold['target_count'] - train_fold['target_count'].median()).abs().mean(),
    })
display(pd.DataFrame(fold_rows).round(2))
print('Terpilih:', report['selected_model'], '+', report['selected_feature_group'])
print('Latih akhir:', report['final_fit_target_years'], '; cek hasil:', report['final_test_year'])

## 7. Hasil 2025: bagus atau belum?

In [ ]:
actual = predictions['target_count']
model_prediction = predictions['prediction_ml']
last_year_prediction = predictions['prediction_last_year']
train_median = panel.loc[panel['target_year'].isin(report['final_fit_target_years']), 'target_count'].median()
median_mae = (actual - train_median).abs().mean()
display(pd.DataFrame([
    {'metode': 'Nilai nyata 2025', 'rata_rata_count': actual.mean(), 'mae': None},
    {'metode': 'Model ML', 'rata_rata_count': model_prediction.mean(), 'mae': report['test_selected_model']['mae']},
    {'metode': 'Tahun sebelumnya', 'rata_rata_count': last_year_prediction.mean(), 'mae': report['test_baseline_last_year']['mae']},
    {'metode': 'Tebak median data latih', 'rata_rata_count': train_median, 'mae': median_mae},
]).round(2))
print(f'Count nyata 2025: median {actual.median():.0f}, rentang {actual.min():.0f}-{actual.max():.0f} per tower')
print(f'Model rata-rata menebak {(model_prediction - actual).mean():.2f} lebih tinggi dari nilai nyata')
display(pd.DataFrame(report['test_by_corridor']).T.round(2))
predictions['absolute_error'] = (model_prediction - actual).abs()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(predictions['target_count'], predictions['prediction_ml'], s=12, alpha=0.55)
limit = max(actual.max(), model_prediction.max())
axes[0].plot([0, limit], [0, limit], 'k--', linewidth=1)
axes[0].set(xlabel='Count nyata 2025', ylabel='Prediksi ML', title='Prediksi vs pengamatan')
predictions.boxplot(column='absolute_error', by='corridor', ax=axes[1], grid=False)
axes[1].set(xlabel='Koridor', ylabel='Kesalahan absolut', title='Kesalahan per koridor')
fig.suptitle('')
plt.tight_layout()
plt.show()
display(predictions.nlargest(10, 'absolute_error')[['corridor', 'tower_key', 'target_count', 'prediction_ml', 'absolute_error']])

## 8. Perbandingan pola dengan PLN

In [ ]:
alignment = report['pln_alignment']
display(pd.DataFrame([{
    'tower_dibandingkan': alignment['eligible_towers'],
    'korelasi_peringkat': alignment['spearman_rho'],
    'tower_sama_di_top_62': alignment['top_62_overlap_count'],
}]).round(2))
print(f"{alignment['excluded_towers']} tower dikecualikan karena nilai Ng workbook tidak cocok dengan data petir 2025")
eligible = predictions[predictions['pln_ng_matches_observed_2025']]
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(eligible['p_strike_annual_pln'], eligible['prediction_p_strike_annual'], s=12, alpha=0.55)
ax.set(xlabel='P_strike_annual PLN', ylabel='Prediksi pada skala PLN', title='Keselarasan paparan tahunan')
plt.tight_layout()
plt.show()

## 9. Ringkasan dan batasan

In [ ]:
model_mae = report['test_selected_model']['mae']
last_year_mae = report['test_baseline_last_year']['mae']
validation_scores = pd.DataFrame(fold_rows).mean(numeric_only=True)
print(f"Validasi 2023-2024: model MAE {validation_scores['mae_model']:.2f}; tebak median MAE {validation_scores['mae_tebak_median_latih']:.2f}. Model dipilih dari sini.")
print(f'Model 2025: MAE {model_mae:.2f}; tahun sebelumnya {last_year_mae:.2f}; tebak median latih {median_mae:.2f}.')
print('Kesimpulan: model belum bagus untuk memperkirakan Count tiap tower pada 2025. Lebih baik dari angka tahun lalu, tetapi kalah dari tebakan median sederhana dan cenderung terlalu tinggi.')
print('Batasan: ini evaluasi ulang 2025, bukan tes buta; 8 riwayat 2020 belum cocok; grounding 2026 tidak dipakai sebagai fitur final; PLN bukan ramalan independen.')